# 24 Local Topic Extraction / Candidate Generation

This notebook performs deterministic, local-only candidate extraction from Phase 23 prepared Bluesky posts.

- No Snowflake access
- No reruns of firehose/hydration/enrichment
- No final matching in this phase


## 1. Load Inputs and Validate Schema

Use the actual prepared Bluesky schema from Phase 23 and fail fast if required fields are missing.


In [1]:
from pathlib import Path
import json
from datetime import datetime, timezone

import pandas as pd

from src.nlp.topic_candidate_generation import (
    REQUIRED_PREPARED_COLUMNS,
    generate_topic_candidates_dataframe,
    summarize_topic_candidates,
)

prepared_path = Path("local/derived/bluesky/bluesky_posts_prepared.parquet")
twitter_ref_path = Path("local/reference_snapshots/twitter_trending/twitter_trending_normalized.parquet")

prepared_df = pd.read_parquet(prepared_path)
print("Prepared rows:", len(prepared_df))
print("Prepared columns:", len(prepared_df.columns))
print("Twitter normalized reference exists:", twitter_ref_path.exists())

missing = [c for c in REQUIRED_PREPARED_COLUMNS if c not in prepared_df.columns]
if missing:
    raise ValueError(f"Missing required prepared columns: {missing}")

prepared_df[REQUIRED_PREPARED_COLUMNS].head(5)


ModuleNotFoundError: No module named 'src'

In [ ]:
prepared_df.dtypes.to_frame("dtype")


## 2. Run Deterministic Candidate Extraction

Balanced extraction strategy: hashtag extraction + filtered 2/3/4-grams + conservative unigram fallback.


In [ ]:
candidates_df = generate_topic_candidates_dataframe(prepared_df, max_candidates_per_post=20)
candidates_df = candidates_df.sort_values(["uri", "candidate_rank_in_post"], kind="stable").reset_index(drop=True)
print("Candidate rows:", len(candidates_df))
print("Posts with candidates:", candidates_df["uri"].nunique())
candidates_df.head(10)


## 3. Candidate Volume and Quality Profiling

Summarize coverage, source/type distribution, and posts with no candidates.


In [ ]:
summary = summarize_topic_candidates(prepared_posts_df=prepared_df, candidates_df=candidates_df)
posts_with_no_candidates = sorted(set(prepared_df["uri"]) - set(candidates_df["uri"]))

summary


In [ ]:
pd.Series(posts_with_no_candidates, name="uri_with_no_candidates")


## 4. Representative Examples

Inspect useful candidates, hashtag-derived candidates, and noisy/edge cases.


In [ ]:
useful_examples = candidates_df[[
    "uri",
    "candidate_source_type",
    "candidate_phrase_raw",
    "candidate_phrase_alnum",
    "candidate_rank_in_post",
]].head(25)
useful_examples


In [ ]:
hashtag_examples = candidates_df.loc[candidates_df["is_hashtag_candidate"], [
    "uri",
    "candidate_phrase_raw",
    "candidate_phrase_clean",
    "candidate_phrase_no_hash",
    "candidate_source_type",
]]
hashtag_examples


In [ ]:
noise_pattern = r"\b(?:macro|youtube|bsky|profile|shorts|com)\b"
noisy_examples = candidates_df.loc[
    candidates_df["candidate_phrase_alnum"].str.contains(noise_pattern, regex=True, na=False),
    ["uri", "candidate_source_type", "candidate_phrase_alnum", "candidate_rank_in_post"],
].head(20)
noisy_examples


## 5. Write Required Outputs

Write full candidate parquet, sample parquet/csv, and optional compact summary JSON.


In [ ]:
local_out = Path("local/derived/bluesky")
sample_out = Path("data/samples")
local_out.mkdir(parents=True, exist_ok=True)
sample_out.mkdir(parents=True, exist_ok=True)

full_out = local_out / "bluesky_topic_candidates.parquet"
sample_parquet_out = sample_out / "bluesky_topic_candidates_sample_1000.parquet"
sample_csv_out = sample_out / "bluesky_topic_candidates_sample_1000.csv"
summary_out = local_out / "bluesky_topic_candidate_summary.json"

candidates_df.to_parquet(full_out, index=False)
candidates_df.head(1000).to_parquet(sample_parquet_out, index=False)
candidates_df.head(1000).to_csv(sample_csv_out, index=False)

summary_payload = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "24_local_topic_extraction_candidate_generation",
    "input_paths": {
        "prepared_bluesky_posts": str(prepared_path),
        "twitter_trending_normalized_reference": str(twitter_ref_path),
    },
    "prepared_schema_columns": prepared_df.columns.tolist(),
    "prepared_row_count": int(len(prepared_df)),
    "candidate_schema_columns": candidates_df.columns.tolist(),
    "candidate_summary": summary,
    "posts_with_no_candidates": posts_with_no_candidates,
    "posts_with_no_candidates_count": int(len(posts_with_no_candidates)),
    "candidate_source_type_counts": {
        str(k): int(v) for k, v in candidates_df["candidate_source_type"].value_counts(dropna=False).items()
    },
    "candidate_token_count_distribution": {
        str(int(k)): int(v) for k, v in candidates_df["candidate_token_count"].value_counts(dropna=False).items()
    },
    "top_candidate_phrases": [
        {"candidate_phrase_alnum": str(k), "count": int(v)}
        for k, v in candidates_df["candidate_phrase_alnum"].value_counts(dropna=False).head(25).items()
    ],
    "output_paths": {
        "full_candidates_parquet": str(full_out),
        "sample_candidates_parquet": str(sample_parquet_out),
        "sample_candidates_csv": str(sample_csv_out),
    },
}
summary_out.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("Wrote:", full_out)
print("Wrote:", sample_parquet_out)
print("Wrote:", sample_csv_out)
print("Wrote:", summary_out)


## 6. Readiness Note

This output is candidate-generation only and is designed to feed Phase 25 post-to-trend matching.
